Load and read the CSV file then display the first few rows of the dataframe.

In [1]:
import pandas as pd
from pathlib import Path

csv_path = Path("../Data/Raw/geothermal_wells_ca.csv")
df = pd.read_csv(csv_path)
df.head()

,X,Y,OBJECTID,WellStatus,WellType,WellSymbol,GISSource,APINumber,GeoDistrict,District,...,Directional,Redrill,SpudDate,ABDdate,CompDate,Lat83,Long83,DatumCode,WellStatusDescription,WellDesignation
0,-13628955.17,4726631.094,1,P,Exploratory Water,ABDN,HUD,1190001,1,6.0,...,NaN,NaN,NaN,NaN,NaN,39.034610,-122.430974,83,Plugged and Abandoned,Wilbur Hot Sprng 1
1,-13628791.09,4726613.754,2,P,Exploratory Water,ABDN,HUD,1190002,1,6.0,...,NaN,NaN,NaN,NaN,NaN,39.034489,-122.429500,83,Plugged and Abandoned,Wilbur Hot Sprgs 1
2,-13629041.78,4726420.285,3,P,Noncommercial Low Temperature,ABDN,HUD,1190003,1,6.0,...,NaN,NaN,NaN,NaN,NaN,39.033139,-122.431752,83,Plugged and Abandoned,Suned/Bailey Min 1
3,-12860481.24,3857606.424,4,I,Observation,IDLE,GPS,2590001,2,1.0,...,NaN,NaN,NaN,NaN,NaN,32.715529,-115.527657,83,Idle,Nowlin Partnersh 1
4,NaN,NaN,5,C,Observation,CANC,,2590002,2,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cancelled,Nowlin Partnersh 2


Display the amount of rows and columns from the data

In [2]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Rows: 4336, Columns: 37


Print the count of each column to see if there are any empty columns

In [3]:
for col, count in df.count().items():
    print(f"  '{col}': {count} values")

  'X': 3440 values
  'Y': 3440 values
  'OBJECTID': 4336 values
  'WellStatus': 4336 values
  'WellType': 4336 values
  'WellSymbol': 4336 values
  'GISSource': 4162 values
  'APINumber': 4336 values
  'GeoDistrict': 4336 values
  'District': 4335 values
  'Confidential': 4336 values
  'ReleaseDate': 0 values
  'CountyAPICode': 4336 values
  'CountyName': 4336 values
  'LeaseName': 4125 values
  'WellNumber': 4336 values
  'OperatorCode': 4336 values
  'OperatorName': 4336 values
  'OperatorStatus': 0 values
  'FieldCode': 0 values
  'FieldName': 2956 values
  'AreaCode': 0 values
  'AreaName': 0 values
  'TownshipSection': 4322 values
  'Township': 4321 values
  'Range': 4321 values
  'BaseMeridian': 4332 values
  'Directional': 0 values
  'Redrill': 0 values
  'SpudDate': 0 values
  'ABDdate': 0 values
  'CompDate': 0 values
  'Lat83': 3440 values
  'Long83': 3440 values
  'DatumCode': 3564 values
  'WellStatusDescription': 4336 values
  'WellDesignation': 4336 values


Inspect rows that have empty latitude and longitude coordinates. Since we won't be able to map them, it's best that we remove these rows so we only work with wells that have coordinates.

In [10]:
missing_lat = df["Lat83"].isna().sum()
missing_lon = df["Long83"].isna().sum()
missing_either = df["Lat83"].isna() | df["Long83"].isna()

print(f"Missing Lat83:            {missing_lat} rows")
print(f"Missing Long83:           {missing_lon} rows")
print(f"Missing Lat83 or Long83:  {missing_either.sum()} rows")
print(f"Total rows:               {len(df)} rows")
print(f"Percentage missing coordinates:  {missing_lat / len(df) * 100:.2f}%")

Missing Lat83:            896 rows
Missing Long83:           896 rows
Missing Lat83 or Long83:  896 rows
Total rows:               4336 rows
Percentage missing coordinates:  20.66%


# We can see that some columns and rows have missing values, which is something we will need to address in our data cleaning process.

See if any of the columns with values displayed above have any missing values (NaN) or empty strings (""). If so, we can decide how to handle those missing values in the next steps.

In [5]:
print(df.isnull().sum())

X                         896
Y                         896
OBJECTID                    0
WellStatus                  0
WellType                    0
WellSymbol                  0
GISSource                 174
APINumber                   0
GeoDistrict                 0
District                    1
Confidential                0
ReleaseDate              4336
CountyAPICode               0
CountyName                  0
LeaseName                 211
WellNumber                  0
OperatorCode                0
OperatorName                0
OperatorStatus           4336
FieldCode                4336
FieldName                1380
AreaCode                 4336
AreaName                 4336
TownshipSection            14
Township                   15
Range                      15
BaseMeridian                4
Directional              4336
Redrill                  4336
SpudDate                 4336
ABDdate                  4336
CompDate                 4336
Lat83                     896
Long83    

Display which columns have no values so they can be removed when we clean the data

In [6]:
empty_columns = df.columns[
    df.apply(lambda col: col.isna().all() or (col.fillna(0) == 0).all())
].tolist()
print(empty_columns)


['ReleaseDate', 'OperatorStatus', 'FieldCode', 'AreaCode', 'AreaName', 'Directional', 'Redrill', 'SpudDate', 'ABDdate', 'CompDate']


Count the unique values of each column. This way we can see if some columns can be used as categorical values or quantitative values

In [7]:
for col, count in df.nunique().items():
    print(f"  '{col}': {count} unique values")

  'X': 3408 unique values
  'Y': 3396 unique values
  'OBJECTID': 4336 unique values
  'WellStatus': 9 unique values
  'WellType': 12 unique values
  'WellSymbol': 11 unique values
  'GISSource': 9 unique values
  'APINumber': 4336 unique values
  'GeoDistrict': 3 unique values
  'District': 4 unique values
  'Confidential': 2 unique values
  'ReleaseDate': 0 unique values
  'CountyAPICode': 21 unique values
  'CountyName': 21 unique values
  'LeaseName': 684 unique values
  'WellNumber': 1801 unique values
  'OperatorCode': 281 unique values
  'OperatorName': 281 unique values
  'OperatorStatus': 0 unique values
  'FieldCode': 0 unique values
  'FieldName': 23 unique values
  'AreaCode': 0 unique values
  'AreaName': 0 unique values
  'TownshipSection': 36 unique values
  'Township': 70 unique values
  'Range': 56 unique values
  'BaseMeridian': 4 unique values
  'Directional': 0 unique values
  'Redrill': 0 unique values
  'SpudDate': 0 unique values
  'ABDdate': 0 unique values
  'C

Print the frequency of each unique value for each column that have values between 1 and 25

In [8]:
for col, count in df.nunique().items():
    if 0 < count and count < 25:
        print(f"'{col}': {count} unique values")
        display(df[col].value_counts().reset_index().rename(columns={'count': 'frequency'}))

'WellStatus': 9 unique values


,WellStatus,frequency
0,P,2038
1,A,1379
2,C,347
3,I,271
4,U,195
5,N,60
6,B,30
7,S,12
8,D,4


'WellType': 12 unique values


,WellType,frequency
0,Temperature Gradient,2008
1,Developmental Steam,618
2,Injection Well,475
3,Developmental Water,405
4,Commercial Low Temperature,253
5,Noncommercial Low Temperature,249
6,Exploratory Water,139
7,Observation,106
8,Exploratory Steam,41
9,Water Well,22


'WellSymbol': 11 unique values


...

,BaseMeridian,frequency
0,MD,2398
1,SB,1932
2,GS,1
3,md,1


'DatumCode': 3 unique values


,DatumCode,frequency
0,83,3431
1,,131
2,27,2


'WellStatusDescription': 10 unique values


,WellStatusDescription,frequency
0,Plugged and Abandoned,2038
1,Active,1379
2,Cancelled,347
3,Idle,271
4,Unknown,195
5,Permitted,60
6,Relinquished,22
7,Suspended,12
8,Rescinded,8
9,Deserted,4
